In [11]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
from xgboost import XGBRegressor

In [12]:
df = pd.read_csv("ml_dataset.csv")

print(df.shape)
print(df.head())

(102468, 58)
  delivery_day  hour  price_value  buy_volume_value  sell_volume_value  \
0   2015-01-01     0        27.40            4849.8             4758.9   
1   2015-01-01     1        27.07            4786.3             4431.1   
2   2015-01-01     2        26.93            4719.0             4328.0   
3   2015-01-01     3        26.83            4692.2             4275.4   
4   2015-01-01     4        26.81            4682.4             4264.5   

   net_volume  flow_DK1_export  flow_DK1_import  flow_GER_export  \
0        90.9              0.0           1131.3              0.0   
1       355.2              0.0           1532.0              0.0   
2       391.0              0.0           1532.0              0.0   
3       416.8              0.0           1532.0              0.0   
4       417.9              0.0           1532.0              0.0   

   flow_GER_import  ...  price_lag_168  price_DK1_lag24  price_NO1_lag24  \
0              0.0  ...            NaN              NaN  

In [13]:
print(df.columns)

Index(['delivery_day', 'hour', 'price_value', 'buy_volume_value',
       'sell_volume_value', 'net_volume', 'flow_DK1_export', 'flow_DK1_import',
       'flow_GER_export', 'flow_GER_import', 'flow_NL_export',
       'flow_NL_import', 'flow_NO1_export', 'flow_NO1_import',
       'flow_NO5_export', 'flow_NO5_import', 'capacity_GER', 'capacity_NL',
       'capacity_NO1', 'capacity_NO5', 'util_GER', 'util_NL', 'util_NO1',
       'util_NO5', 'capacity_missing_DK1', 'capacity_missing_GER',
       'capacity_missing_NL', 'capacity_missing_NO1', 'capacity_missing_NO5',
       'capacity_zero_DK1', 'capacity_zero_GER', 'capacity_zero_NL',
       'capacity_zero_NO1', 'capacity_zero_NO5', 'price_DK1', 'price_NO1',
       'price_NO5', 'total_exports', 'total_imports', 'net_flow',
       'day_of_week', 'month', 'day_of_year', 'day_sin', 'day_cos', 'year_sin',
       'year_cos', 'price_lag_24', 'price_lag_168', 'price_DK1_lag24',
       'price_NO1_lag24', 'price_NO5_lag24', 'price_DK1_lag168',
       

In [ ]:
df["delivery_day"] = pd.to_datetime(df["delivery_day"])

#optionally filter out data before 2025-01-01
df = df[df["delivery_day"] >= "2020-01-01"]

# Target: price 24 hours ahead
df = df.sort_values(["delivery_day", "hour"]).reset_index(drop=True)

df["target_price_next_day"] = df["price_value"].shift(-24)

# Drop rows without a future price (last 24 rows)
df = df.dropna(subset=["target_price_next_day"])

print(df.shape)
df[["delivery_day", "hour", "price_value", "target_price_next_day"]].head(10)

(14758, 59)


,delivery_day,hour,price_value,target_price_next_day
0,2025-01-01,0,16.35,27.11
1,2025-01-01,1,14.05,26.80
2,2025-01-01,2,12.39,26.15
3,2025-01-01,3,13.21,27.38
4,2025-01-01,4,12.23,29.36
5,2025-01-01,5,6.73,31.98
6,2025-01-01,6,15.37,37.82
7,2025-01-01,7,16.11,88.61
8,2025-01-01,8,16.55,115.95
9,2025-01-01,9,17.95,119.84


In [19]:
drop_cols = [
    "delivery_day",
]

X = df.drop(columns=drop_cols + ["target_price_next_day"])
y = df["target_price_next_day"]

print("Features:", len(X.columns))

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(f"Train rows: {len(X_train):,}")
print(f"Test rows : {len(X_test):,}")

Features: 57
Train rows: 11,806
Test rows : 2,952


In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

print("Training...")
model.fit(X_train, y_train)

print("Predicting...")
pred = model.predict(X_test)
print("done")

In [ ]:
rmse = root_mean_squared_error(y_test, pred)

print(f"RMSE = {rmse:.4f}")

r2 = r2_score(
    y_test,
    pred
)

print("R²:", r2)

rmse = root_mean_squared_error(
y_test,
X_test["price_lag24"]
)
print(f"price_lag24 RMSE: {rmse:.4f}")

RMSE = 25.7843
R²: 0.6138610003815266
price_value_lag24 RMSE: 36.1600


In [ ]:
model = XGBRegressor(
        n_estimators=500,
        max_depth=5,
        min_child_weight=5,
        learning_rate=0.01,
        objective="reg:squarederror"
    )

print("Training...")
model.fit(X_train, y_train)

print("Predicting...")
pred = model.predict(X_test)
print("done")

Training...
Predicting...
done


In [25]:
rmse = root_mean_squared_error(y_test, pred)

print(f"RMSE = {rmse:.4f}")

r2 = r2_score(
    y_test,
    pred
)

print("R²:", r2)

rmse = root_mean_squared_error(
y_test,
X_test["price_lag_24"]
)
print(f"price_lag_24 RMSE: {rmse:.4f}")

RMSE = 31.0471
R²: 0.466254679266887
price_lag_24 RMSE: 39.3918


In [26]:
importance = (
pd.DataFrame({
"feature": X.columns,
"importance": model.feature_importances_
})
.sort_values("importance", ascending=False)
)
print(importance.head(20))

              feature  importance
1         price_value    0.251760
34          price_NO1    0.241395
35          price_NO5    0.032789
39        day_of_week    0.029451
33          price_DK1    0.026406
43            day_cos    0.023952
41        day_of_year    0.023456
40              month    0.018522
11    flow_NO1_export    0.018042
9      flow_NL_export    0.016229
10     flow_NL_import    0.015973
55      load_forecast    0.015614
53   price_NO5_lag168    0.015511
44           year_sin    0.015349
51   price_DK1_lag168    0.015293
5     flow_DK1_export    0.013656
52   price_NO1_lag168    0.013552
45           year_cos    0.013312
3   sell_volume_value    0.012907
46       price_lag_24    0.012775
